Static Circuit : Before Grover Vs After Grover 
Dynamic Circuit : Before Grover Vs After Grovera

Metrics:
1. Probability of obtaining a marked optimal trajectory
2. Amplification gain relative to no-Grover circuit

Benchmark:
T-143 and T-151 are the two marked maximum-return trajectories.

Results report -->  mean +/- standard deviation with confidence intervals

In [ ]:
# importing packages
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit, transpile
from qiskit_aer.backends import AerSimulator
from qiskit.circuit.library import RYGate
from qiskit.circuit import Parameter
import numpy as np
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
from typing import List
import pandas as pd
from matplotlib.lines import Line2D
import os
import pickle
import time

from scipy.stats import t

In [ ]:
# ============================================================
# 1. SETTINGS
# ============================================================

SHOTS = 1024
# SEEDS = list(range(10))
# SEEDS = [11, 101, 503, 1009, 2003, 3001, 4001, 5003, 7001, 9001]
# rng = np.random.default_rng(20261011)

# SEEDS = rng.choice(
#     1_000_000,
#     size=30,
#     replace=False
# ).tolist()

SEEDS = [
    670445, 805168, 539147, 725762, 449165,
    577976, 95441, 320590, 269738, 441644,
    425136, 434180, 462454, 427846, 319328,
    327223, 126166, 490229, 878188, 23132,
    693954, 937355, 771686, 675450, 644523,
    56823, 682125, 386350, 648423, 817822
]

print(SEEDS)

T143 = "1000111101111111100101000"  # original bit order directly after measurement
T151 = "1000111111111111100101000" #original bit order directly after measurement
TARGETS = [T143, T151]

In [ ]:
# Classical environment (Modified Transition)

class ClassicToyEnv:
    
    # Constructor. 
    # Param MaxSteps: Maximum number of environment/agent interactions allowed for an episode
    def __init__(self, MaxSteps=np.Inf):
        
        self.MaxSteps= MaxSteps # Maximum number of steps before ending an episode
        
        # ========================================
        # MDP definition
        self.nS= 4 # Number of MDP states
        self.nA= 2 # Number of MDP actions
        self.nR= 4 # Number of possible rewards
        
        # Transition function
        self.T= np.zeros((self.nS, self.nA, self.nS))
        self.T[0, 0, 1]= 0.6
        self.T[0, 0, 2]= 0.4
        self.T[0, 1, 0]= 0.1 #here
        self.T[0, 1, 1]= 0.9 #here
        self.T[1, 0, 0]= 0.2 
        self.T[1, 0, 1]= 0.8
        self.T[1, 1, 2]= 0.7
        self.T[1, 1, 3]= 0.3
        self.T[2, 0, 0]= 0.3
        self.T[2, 0, 2]= 0.7
        self.T[2, 1, 1]= 0.2
        self.T[2, 1, 3]= 0.8
        self.T[3, 0, 2]= 0.6
        self.T[3, 0, 3]= 0.4
        self.T[3, 1, 3]= 1.0
        
        # Current state and step
        self.currentState= None
        self.currentStep= None
        
    # returns the number of states in the environment
    def numberOfStates(self):
        return self.nS
       
    # returns the number of states in the environment
    def numberOfActions(self):
        return self.nA
    
    # Returns the transition probability from state s and action a to state sp
    def transitionProb(self, s, a, sp):
        return self.T[s, a, sp]

**Static QMDP before Grover and After Grover**

In [ ]:
# Quantum Environment for Two State Transitions with Return _ STATIC QMDP BEFORE Grover

class QuantumToyEnv_Static(ClassicToyEnv):
    
    # Constructor. 
    # Param MaxSteps: Maximum number of environment/agent interactions allowed for an episode
    def __init__(self, MaxSteps=np.Inf):
        super().__init__(MaxSteps)
    
        # ========================================
        # Quantum MDP representation 
        self.nqS= 2 # Number of qubits for state representation= log2(nS)
        self.nqA= 1 # Number of qubits required for action representation= log2(nA)
        self.nqR= 2 # Number of qubits for reward representation= log2(nR)
        self.nqRt = 4 # Number of qubits for return

        # Quantum Registers for MDP 2-state transition
        self.qS0= QuantumRegister(self.nqS, name='t0_qS') # Quantum Register to store input environment states
        self.qSp0= QuantumRegister(self.nqS, name='t0_qSp') # Quantum Register to store output environment states
        self.qA0= QuantumRegister(self.nqA, name='t0_qA') # Quantum Register to store actions
        self.qR0= QuantumRegister(self.nqR, name='t0_qR') # Quantum Register to store rewards
        
        self.qS1= QuantumRegister(self.nqS, name='t1_qS') # Quantum Register to store input environment states
        self.qSp1= QuantumRegister(self.nqS, name='t1_qSp') # Quantum Register to store output environment states
        self.qA1= QuantumRegister(self.nqA, name='t1_qA') # Quantum Register to store actions
        self.qR1= QuantumRegister(self.nqR, name='t1_qR') # Quantum Register to store rewards
        
        self.qS2= QuantumRegister(self.nqS, name='t2_qS') # Quantum Register to store input environment states
        self.qSp2= QuantumRegister(self.nqS, name='t2_qSp') # Quantum Register to store output environment states
        self.qA2= QuantumRegister(self.nqA, name='t2_qA') # Quantum Register to store actions
        self.qR2= QuantumRegister(self.nqR, name='t2_qR') # Quantum Register to store rewards
        
        # Quantum Registers for Return
        self.qRt= QuantumRegister(self.nqRt, name='qRt') # Quantum Register to store return
        
        # Classical Registers
        self.cRt= ClassicalRegister(4, name='cR') # Classical Register to store measured return

        self.qc= QuantumCircuit(self.qS0, self.qA0, self.qSp0, self.qR0,
                                self.qS1, self.qA1, self.qSp1, self.qR1,
                                self.qS2, self.qA2, self.qSp2, self.qR2,
                                self.qRt)

        # for t = 0 , initial step      
        # Us implementation: Appends Us to self.qc
        self.__Us_0(self.qc, self.qS0)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA0)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS0, self.qA0, self.qSp0)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp0, self.qR0)
        
        # for t = 1
        self.__Us(self.qc, self.qSp0, self.qS1)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA1)

        # # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS1, self.qA1, self.qSp1)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp1, self.qR1)
        
        # for t = 2
        self.__Us(self.qc, self.qSp1, self.qS2)

        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA2)

        # # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS2, self.qA2, self.qSp2)

        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp2, self.qR2)
        

    # Implementation of the QSample encoding
    def __QSampleNextState(self, probs, qc, qS, qA, qSp, control_s, control_a, control_sp= '', currentQ=0):
        
        all_controls= control_s + control_a + control_sp
        
        # Find control qubits
        controlsS= []
        for i in range(len(qS)):
            controlsS.append(qS[i])
        controlsA= []
        for i in range(len(qA)):
            controlsA.append(qA[i])
        controlsSp= []
        for i in range(currentQ):
            controlsSp.append(qSp[i]) 
        
        # Base case: End of recursion
        if currentQ>=len(qSp):
            return
        
        else: # General case
        
            # compute prob(qS[currentQ | qS, qA, qS[:currentQ-1]])
            add_prob= np.sum(probs)
            if add_prob > 0:
                current_prob= np.sum(probs[len(probs)//2:])/add_prob
                angle= 2*np.arcsin(np.sqrt(current_prob))
                
                # Make custom controlled CRY gate 
                cry= RYGate(angle).control(num_ctrl_qubits=len(all_controls), 
                                             ctrl_state= all_controls)
                
                qc.append(cry, [*controlsSp, *controlsA, *controlsS, qSp[currentQ]])
        
            self.__QSampleNextState(probs[:len(probs)//2], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'0', currentQ=currentQ+1)
            self.__QSampleNextState(probs[len(probs)//2:], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'1', currentQ=currentQ+1)
      
    # for state in superposition
    def __Us_0(self, qc, qS):
        for i in range(len(qS)):
            qc.h(qS[i])
        qc.barrier()
        
    def __Us(self, qc, qSp, qS):   
        for i in range(len(qS)):
           if i == 1:
             qc.cx(qSp[i],qS[i-1])
           else:
            qc.cx(qSp[i],qS[i+1])
        qc.barrier()
        
    def __Ua(self, qc, qA):
        for i in range(len(qA)):
            qc.h(qA[i])
        qc.barrier()
    
    # Circuit to implement transition function
    def __Ut(self, qc, qS, qA, qSp):
        
        for s in range(self.nS): 
            for a in range(self.nA):
                
                # Conditionals for input state s
                binS= bin(s)[2:][::-1]
                while len(binS)<self.nqS:
                    binS+= '0'
                binS= binS[::-1]
                
                # Conditionals for input action a
                binA= bin(a)[2:][::-1]
                while len(binA)<self.nqA:
                    binA+= '0'
                binA= binA[::-1]
        
                # Transition Function P(s,a)
                probs= self.T[s, a]
                self.__QSampleNextState(probs, self.qc, qS, qA, qSp, 
                        control_s=binS, 
                        control_a=binA)
                self.qc.barrier()

    
    # Circuit to implement reward function
    def __Ur(self, qc, qSp, qR):
        
        # Hard-coded for example MDP
        for i in range(len(qSp)):
            qc.cx(qSp[i], qR[i])
        self.qc.barrier()    
        
####################################### Functions #############
    
# Circuit to implement return function
def __URt(qc,t0_qR0, t0_qR1, t1_qR0,t1_qR1,t2_qR0,t2_qR1,qRt_0,qRt_1,qRt_2,qRt_3):
  ## return oracle

  #### for LSB output qubit q6
    qc.cx(t0_qR0,qRt_0)
    qc.cx(t1_qR0,qRt_0)
    qc.cx(t2_qR0,qRt_0)
    qc.barrier()

  #### for output qubit q7
    qc.cx(t0_qR1,qRt_1)
    qc.cx(t1_qR1,qRt_1)
    qc.cx(t2_qR1,qRt_1)
  ## carry-in part
    qc.ccx(t0_qR0,t1_qR0,qRt_1)
    qc.ccx(t0_qR0,t2_qR0,qRt_1)
    qc.ccx(t1_qR0,t2_qR0,qRt_1)
    qc.barrier()


  #### for output qubit q8 (carry-out qbit)
  # when no carry-in bit from LSB
    qc.ccx(t0_qR1,t1_qR1,qRt_2)
    qc.ccx(t0_qR1,t2_qR1,qRt_2)
    qc.ccx(t1_qR1,t2_qR1,qRt_2)

  # when carry-in bit from LSB
    qc.mcx([t0_qR0,t1_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_2])
    qc.mcx([t0_qR0,t2_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_2])
    qc.mcx([t1_qR0,t2_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_2])
  ## carry-in and single one qubit
    qc.mcx([t0_qR0,t1_qR0,t0_qR1],[qRt_2])
    qc.mcx([t0_qR0,t1_qR0,t1_qR1],[qRt_2])
    qc.mcx([t0_qR0,t1_qR0,t2_qR1],[qRt_2])
    
    qc.mcx([t0_qR0,t2_qR0,t0_qR1],[qRt_2])
    qc.mcx([t0_qR0,t2_qR0,t1_qR1],[qRt_2])
    qc.mcx([t0_qR0,t2_qR0,t2_qR1],[qRt_2])
    
    qc.mcx([t1_qR0,t2_qR0,t0_qR1],[qRt_2])
    qc.mcx([t1_qR0,t2_qR0,t1_qR1],[qRt_2])
    qc.mcx([t1_qR0,t2_qR0,t2_qR1],[qRt_2])
    
    qc.mcx([t0_qR0,t1_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_2])
    qc.mcx([t0_qR0,t2_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_2])
    qc.mcx([t1_qR0,t2_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_2])
    
    qc.barrier()

  ### for output qubit q9 (carry-out qubit)
    qc.mcx([t0_qR0,t1_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_3])
    qc.mcx([t0_qR0,t2_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_3])
    qc.mcx([t1_qR0,t2_qR0,t0_qR1,t1_qR1,t2_qR1],[qRt_3])
    qc.barrier()

In [ ]:
# Static QMDP Oracle and Diffuser

# oracle
qreg = QuantumRegister(25, name ='g')
oracle = QuantumCircuit(qreg, name = 'Grover')
oracle.x([23,22,21,1,0])
oracle.h(24)
oracle.mcx([23,22,21,18,17,1,0],24)
oracle.h(24)
oracle.x([23,22,21,1,0])
# oracle.draw('mpl')

# customized diffuser
qreg = QuantumRegister(25, name ='d')
diffuser = QuantumCircuit(qreg, name = 'Diffuser')
nqubits = diffuser.num_qubits

diffuser.x([nqubits-25,nqubits-24])

diffuser.h([nqubits-25,nqubits-24,nqubits-23])
diffuser.x([nqubits-25,nqubits-24,nqubits-23])
diffuser.h(nqubits-19)
diffuser.mcx([nqubits-25,nqubits-24,nqubits-23],nqubits-19)
diffuser.h(nqubits-19)
diffuser.x([nqubits-25,nqubits-24,nqubits-23])
diffuser.h([nqubits-25,nqubits-24,nqubits-23])
diffuser.barrier()

diffuser.h([nqubits-18,nqubits-17,nqubits-16])
diffuser.x([nqubits-18,nqubits-17,nqubits-16])
diffuser.h(nqubits-12)
diffuser.mcx([nqubits-18,nqubits-17,nqubits-16],nqubits-12)
diffuser.h(nqubits-12)
diffuser.h(nqubits-13)
diffuser.mcx([nqubits-18,nqubits-17,nqubits-16],nqubits-13)
diffuser.h(nqubits-13)
diffuser.x([nqubits-18,nqubits-17,nqubits-16])
diffuser.h([nqubits-18,nqubits-17,nqubits-16])
diffuser.barrier()

diffuser.h([nqubits-11,nqubits-10,nqubits-9])
diffuser.x([nqubits-11,nqubits-10,nqubits-9])
diffuser.h(nqubits-5)
diffuser.mcx([nqubits-11,nqubits-10,nqubits-9],nqubits-5)
diffuser.h(nqubits-5)
diffuser.h(nqubits-6)
diffuser.mcx([nqubits-11,nqubits-10,nqubits-9],nqubits-6)
diffuser.h(nqubits-6)
diffuser.x([nqubits-11,nqubits-10,nqubits-9])
diffuser.h([nqubits-11,nqubits-10,nqubits-9])
diffuser.barrier()

diffuser.h([nqubits-1,nqubits-2,nqubits-3,nqubits-4])
diffuser.mcx([nqubits-19,nqubits-12,nqubits-13,nqubits-5,nqubits-6],nqubits-1)
diffuser.mcx([nqubits-19,nqubits-12,nqubits-13,nqubits-5,nqubits-6],nqubits-2)
diffuser.mcx([nqubits-19,nqubits-12,nqubits-13,nqubits-5,nqubits-6],nqubits-3)
diffuser.mcx([nqubits-19,nqubits-12,nqubits-13,nqubits-5,nqubits-6],nqubits-4)
diffuser.h([nqubits-1,nqubits-2,nqubits-3,nqubits-4])

# diffuser.draw('mpl')

**Dynamic QMDP before Grover and After Grover**

In [ ]:
# DYNAMIC QMDP before Grover
# Quantum Environment for Two State Transitions with Return
# Storing Reward qubits into Ancillary qubits before mid-circuit measurement

class QuantumToyEnv_Dynamic(ClassicToyEnv):
    
    # Constructor. 
    # Param MaxSteps: Maximum number of environment/agent interactions allowed for an episode
    def __init__(self, MaxSteps=np.inf):
        super().__init__(MaxSteps)
    
        # ========================================
        # Quantum MDP representation 
        self.nqS= 2 # Number of qubits for state representation= log2(nS)
        self.nqA= 1 # Number of qubits required for action representation= log2(nA)
        self.nqR= 2 # Number of qubits for reward representation= log2(nR)
        self.nqRt = 4 # Number of qubits for return
        self.nqAnci = 2 # Number of ancillary qubits

        # Quantum Registers for MDP 2-state transition 
        self.qS= QuantumRegister(self.nqS, name='qState') # Quantum Register to store input environment states
        self.qSp= QuantumRegister(self.nqS, name='qNextState') # Quantum Register to store output environment states
        self.qA= QuantumRegister(self.nqA, name='qAction') # Quantum Register to store actions
        self.qR= QuantumRegister(self.nqR, name='qReward') # Quantum Register to store rewards
        
        self.t0_qAnci = QuantumRegister(self.nqAnci, name = 't0_qAnci') # ancillary qubits to store the timestep 0 reward
        self.t1_qAnci = QuantumRegister(self.nqAnci, name = 't1_qAnci') # ancillary qubits to store the timestep 1 reward
        
        # Quantum Registers for Return
        self.qRt= QuantumRegister(self.nqRt, name='qRt') # Quantum Register to store return
        
        # Classical Registers
        self.t0_cReg= ClassicalRegister(7, name='t0_cReg') # Classical Register to store measured timestep 0
        self.t1_cReg= ClassicalRegister(7, name='t1_cReg') # Classical Register to store measured timestep 1
        self.t2_cReg= ClassicalRegister(7, name='t2_cReg') # Classical Register to store measured timestep 2
        self.cRt= ClassicalRegister(4, name='cR') # Classical Register to store measured return
        
        # Quantum Circuit to store a cycle of an MDP
        self.qc= QuantumCircuit(self.qS, self.qA, self.qSp, self.qR, self.t0_qAnci,  self.t1_qAnci, self.qRt, 
                                           self.t0_cReg, self.t1_cReg, self.t2_cReg, self.cRt)
        
        # ----------------------------------------------------------------------------------------------------------------
        # for t = 0 , Initial step      
        # Us implementation: Appends Us to self.qc
        self.__Us_0(self.qc, self.qS)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
        
        # Uanci implementation: Store the reward at the ancillary qubits
        self.__Uanci(self.qc, self.qR, self.t0_qAnci) 
        
        self.qc.barrier()
        
        """ Mid-ciruit Measurement at timestep 0 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.t0_cReg[0], self.t0_cReg[1], self.t0_cReg[2], self.t0_cReg[3], self.t0_cReg[4], self.t0_cReg[5], self.t0_cReg[6]])
        
        self.qc.barrier()
              
        # ------------------------------------------------------------------------------------------------------------
        # for t = 1     
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 0 to t = 1. """       
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)
        
        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
        
        # Uanci implementation: Store the reward at the ancillary qubits
        self.__Uanci(self.qc, self.qR, self.t1_qAnci) 
        
        self.qc.barrier()
         
        """ Mid-ciruit Measurement at timestep 1 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.t1_cReg[0], self.t1_cReg[1], self.t1_cReg[2], self.t1_cReg[3], self.t1_cReg[4], self.t1_cReg[5], self.t1_cReg[6]])
        
        self.qc.barrier()
               
        # -----------------------------------------------------------------------------------------------------------------
        # for t = 2
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 1 to t = 2. """    
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)

        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)

        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)

        self.qc.barrier()
        
        # -----------------------------------------------------------------------------------------------------------------
        # Return
        self.__UReturn(self.qc, self.t0_qAnci, self.t1_qAnci, self.qR, self.qRt)
        
        """ Mid-ciruit Measurement at timestep 2 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.t2_cReg[0], self.t2_cReg[1], self.t2_cReg[2], self.t2_cReg[3], self.t2_cReg[4], self.t2_cReg[5], self.t2_cReg[6]])       
       
        self.qc.barrier()
        
        self.qc.measure([self.qRt[0], self.qRt[1], self.qRt[2], self.qRt[3]], 
                               [self.cRt[0], self.cRt[1], self.cRt[2], self.cRt[3]])     
        
    # Implementation of the QSample encoding
    def __QSampleNextState(self, probs, qc, qS, qA, qSp, control_s, control_a, control_sp= '', currentQ=0):
        
        all_controls= control_s + control_a + control_sp
        
        # Find control qubits
        controlsS= []
        for i in range(len(qS)):
            controlsS.append(qS[i])
        controlsA= []
        for i in range(len(qA)):
            controlsA.append(qA[i])
        controlsSp= []
        for i in range(currentQ):
            controlsSp.append(qSp[i])
        
        # Base case: End of recursion
        if currentQ>=len(qSp):
            return
        
        else: # General case
        
            # compute prob(qS[currentQ | qS, qA, qS[:currentQ-1]])
            add_prob= np.sum(probs)
            if add_prob > 0:
                current_prob= np.sum(probs[len(probs)//2:])/add_prob
                angle= 2*np.arcsin(np.sqrt(current_prob))
                
                # Make custom controlled CRY gate 
                cry= RYGate(angle).control(num_ctrl_qubits=len(all_controls), 
                                             ctrl_state= all_controls)
                
                qc.append(cry, [*controlsSp, *controlsA, *controlsS, qSp[currentQ]])
        
            self.__QSampleNextState(probs[:len(probs)//2], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'0', currentQ=currentQ+1)
            self.__QSampleNextState(probs[len(probs)//2:], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'1', currentQ=currentQ+1)
    

    # Circuit to set input environment state  
    # for state in superposition
    def __Us_0(self, qc, qS):
        for i in range(len(qS)):
            qc.h(qS[i])

    def __Us(self, qc, qSp, qS):   
        for i in range(len(qS)):
           if i == 1:
             qc.cx(qSp[i],qS[i-1])
           else:
            qc.cx(qSp[i],qS[i+1])
       
    # Circuit to set input action    
    def __Ua(self, qc, qA):
        for i in range(len(qA)):
            qc.h(qA[i])
    
    # Circuit to implement transition function
    def __Ut(self, qc, qS, qA, qSp):
        
        for s in range(self.nS): 
            for a in range(self.nA):
                
                # Conditionals for input state s
                binS= bin(s)[2:][::-1]
                while len(binS)<self.nqS:
                    binS+= '0'
                binS= binS[::-1]
                
                # Conditionals for input action a
                binA= bin(a)[2:][::-1]
                while len(binA)<self.nqA:
                    binA+= '0'
                binA= binA[::-1]
        
                # Transition Function P(s,a)
                probs= self.T[s, a]
                self.__QSampleNextState(probs, self.qc, qS, qA, qSp, 
                        control_s=binS, 
                        control_a=binA)

    # Circuit to implement reward function
    def __Ur(self, qc, qSp, qR):
        for i in range(len(qSp)):
            qc.cx(qSp[i], qR[i])

    # Circuit to store the reward at the ancillary qubits
    def __Uanci(self, qc, qR, qAnci):
        for i in range(len(qR)):
            qc.cx(qR[i], qAnci[i])       


    # Circuit to store the reward at the ancillary qubits
    def __UReturn(self, qc, t0_qReward, t1_qReward, qReward, qReturn):
      ## For Return qubit[0]
      qc.cx(t0_qReward[1], qReturn[0])
      qc.cx(t1_qReward[1], qReturn[0])
      qc.cx(qReward[1], qReturn[0])
      
      qc.barrier()

      ## For Return qubit[1]
      qc.cx(t0_qReward[0], qReturn[1])
      qc.cx(t1_qReward[0], qReturn[1])
      qc.cx(qReward[0], qReturn[1])
      # For carry-in part
      qc.ccx(t0_qReward[1], t1_qReward[1], qReturn[1])
      qc.ccx(t0_qReward[1], qReward[1], qReturn[1])
      qc.ccx(t1_qReward[1], qReward[1], qReturn[1])
      qc.barrier()
      
      # For Return qubit [2]
      # when no carry-in bit from LSB
      qc.ccx(t0_qReward[0],  t1_qReward[0], qReturn[2])
      qc.ccx(t0_qReward[0], qReward[0],qReturn[2])
      qc.ccx(t1_qReward[0], qReward[0] ,qReturn[2])

      # when carry-in bit from LSB
      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      ## carry-in and single one qubit
      qc.mcx([t0_qReward[1], t1_qReward[1],t0_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], t1_qReward[1], t1_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], t1_qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1],  t1_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t1_qReward[1], qReward[1],t0_qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t1_qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[2]])

      qc.barrier()

      ### for  Return qubit [3] (carry-out qubit)
      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[3]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[3]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[3]])
      qc.barrier()


In [ ]:
# DYNAMIC QMDP after Grover
# Quantum Environment for Two State Transitions with Return
#  qs --> reset + hadamard 
# Reward qubts store to Ancillary qubits before mid-circuit measurement

class QuantumToyEnv_Dynamic_G(ClassicToyEnv):
    
    # Constructor. 
    # Param MaxSteps: Maximum number of environment/agent interactions allowed for an episode
    def __init__(self, MaxSteps=np.Inf):
        super().__init__(MaxSteps)
    
        # ========================================
        # Quantum MDP representation 
        self.nqS= 2 # Number of qubits for state representation= log2(nS)
        self.nqA= 1 # Number of qubits required for action representation= log2(nA)
        self.nqR= 2 # Number of qubits for reward representation= log2(nR)
        self.nqRt = 4 # Number of qubits for return
        self.nqAnci = 2 # Number of ancillary qubits

        # Quantum Registers for MDP 2-state transition 
        self.qS= QuantumRegister(self.nqS, name='qState') # Quantum Register to store input environment states
        self.qSp= QuantumRegister(self.nqS, name='qNextState') # Quantum Register to store output environment states
        self.qA= QuantumRegister(self.nqA, name='qAction') # Quantum Register to store actions
        self.qR= QuantumRegister(self.nqR, name='qReward') # Quantum Register to store rewards
        
        self.t0_qAnci = QuantumRegister(self.nqAnci, name = 't0_qAnci') # ancillary qubits to store the timestep 0 reward
        self.t1_qAnci = QuantumRegister(self.nqAnci, name = 't1_qAnci') # ancillary qubits to store the timestep 1 reward
        
        # Quantum Registers for Return
        self.qRt= QuantumRegister(self.nqRt, name='qRt') # Quantum Register to store return
        
        # Classical Registers
        self.cReg= ClassicalRegister(25, name='cReg') # Classical Register to store measured three timesteps and return

        # Quantum Circuit to store a cycle of an MDP
        self.qc= QuantumCircuit(self.qS, self.qA, self.qSp, self.qR, self.t0_qAnci,  self.t1_qAnci, self.qRt, self.cReg) 
        
        # ----------------------------------------------------------------------------------------------------------------
        # for t = 0 , Initial step      
        # Us implementation: Appends Us to self.qc
        self.__Us_0(self.qc, self.qS)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
        
        # Uanci implementation: Store the reward at the ancillary qubits
        self.__Uanci(self.qc, self.qR, self.t0_qAnci) 
        
        self.qc.barrier()
                
        """ Mid-ciruit Measurement at timestep 0 """
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.cReg[0], self.cReg[1], self.cReg[2], self.cReg[3], self.cReg[4], self.cReg[5], self.cReg[6]])
        
        self.qc.barrier()
        
        # ------------------------------------------------------------------------------------------------------------
        # for t = 1     
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 0 to t = 1. """       
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)
        
        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
        
       # Uanci implementation: Store the reward at the ancillary qubits
        self.__Uanci(self.qc, self.qR, self.t1_qAnci) 
        
        self.qc.barrier()
         
        """ Mid-ciruit Measurement at timestep 1 """ 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.cReg[7], self.cReg[8], self.cReg[9], self.cReg[10], self.cReg[11], self.cReg[12], self.cReg[13]])       
        self.qc.barrier()
                
        # -----------------------------------------------------------------------------------------------------------------
        # for t = 2
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 1 to t = 2. """    
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)

        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)

        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)    

        self.qc.barrier()
        
        # -----------------------------------------------------------------------------------------------------------------
        # Return
        self.__UReturn(self.qc, self.t0_qAnci, self.t1_qAnci, self.qR, self.qRt)
        
        self.qc.barrier()
      
        #  -------------------------------------------------------------------------------------------------------------------
        # # QRL (Grover's search for '1000' starting state)
        self.iterations = 1
        for i in range( self.iterations):
         """oracle (state, next state, reward at t=0, reward at t=1, reward at t=2, return)"""
         self.__UOracle(self.qc, self.qS,self.qSp, self.t0_qAnci, self.t1_qAnci, self.qR, self.qRt[0], self.qRt[1], self.qRt[2], self.qRt[3])
         self.qc.barrier()
         
         """diffuser  (state, action, next state, reward at t=0, reward at t=1, reward at t=2, return)"""
         self.__UDiffuser(self.qc, self.qS, self.qA, self.qSp, self.t0_qAnci, self.t1_qAnci, self.qR, self.qRt[0], self.qRt[1], self.qRt[2], self.qRt[3])
         self.qc.barrier()
         
        # end of iteration loop   
    
        """ Mid-ciruit Measurement at timestep 2  and Return"""
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.cReg[14], self.cReg[15], self.cReg[16], self.cReg[17], self.cReg[18], self.cReg[19], self.cReg[20]])     
        
        self.qc.measure([self.qRt[0], self.qRt[1], self.qRt[2], self.qRt[3]], 
                               [self.cReg[21], self.cReg[22], self.cReg[23], self.cReg[24]])     
        
    # Implementation of the QSample encoding
    def __QSampleNextState(self, probs, qc, qS, qA, qSp, control_s, control_a, control_sp= '', currentQ=0):
        
        all_controls= control_s + control_a + control_sp
        
        # Find control qubits
        controlsS= []
        for i in range(len(qS)):
            controlsS.append(qS[i])
        controlsA= []
        for i in range(len(qA)):
            controlsA.append(qA[i])
        controlsSp= []
        for i in range(currentQ):
            controlsSp.append(qSp[i])
        
        # Base case: End of recursion
        if currentQ>=len(qSp):
            return
        
        else: # General case
        
            # compute prob(qS[currentQ | qS, qA, qS[:currentQ-1]])
            add_prob= np.sum(probs)
            if add_prob > 0:
                current_prob= np.sum(probs[len(probs)//2:])/add_prob
                angle= 2*np.arcsin(np.sqrt(current_prob))
                
                # Make custom controlled CRY gate 
                cry= RYGate(angle).control(num_ctrl_qubits=len(all_controls), 
                                             ctrl_state= all_controls)
                
                qc.append(cry, [*controlsSp, *controlsA, *controlsS, qSp[currentQ]])
        
            self.__QSampleNextState(probs[:len(probs)//2], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'0', currentQ=currentQ+1)
            self.__QSampleNextState(probs[len(probs)//2:], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'1', currentQ=currentQ+1)
    

    # Circuit to set input environment state  
    # for state in superposition
    def __Us_0(self, qc, qS):
        for i in range(len(qS)):
            qc.h(qS[i])

    def __Us(self, qc, qSp, qS):   
        for i in range(len(qS)):
           if i == 1:
             qc.cx(qSp[i],qS[i-1])
           else:
            qc.cx(qSp[i],qS[i+1])
       
    # Circuit to set input action    
    def __Ua(self, qc, qA):
        for i in range(len(qA)):
            qc.h(qA[i])
    
    # Circuit to implement transition function
    def __Ut(self, qc, qS, qA, qSp):
        
        for s in range(self.nS): 
            for a in range(self.nA):
                
                # Conditionals for input state s
                binS= bin(s)[2:][::-1]
                while len(binS)<self.nqS:
                    binS+= '0'
                binS= binS[::-1]
                
                # Conditionals for input action a
                binA= bin(a)[2:][::-1]
                while len(binA)<self.nqA:
                    binA+= '0'
                binA= binA[::-1]
        
                # Transition Function P(s,a)
                probs= self.T[s, a]
                self.__QSampleNextState(probs, self.qc, qS, qA, qSp, 
                        control_s=binS, 
                        control_a=binA)

    # Circuit to implement reward function
    def __Ur(self, qc, qSp, qR):
        for i in range(len(qSp)):
            qc.cx(qSp[i], qR[i])

    # Circuit to store the reward at the ancillary qubits
    def __Uanci(self, qc, qR, qAnci):
        for i in range(len(qR)):
            qc.cx(qR[i], qAnci[i])       


    # Circuit to store the reward at the ancillary qubits
    def __UReturn(self, qc, t0_qReward, t1_qReward, qReward, qReturn):
      ## For Return qubit[0]
      qc.cx(t0_qReward[1], qReturn[0])
      qc.cx(t1_qReward[1], qReturn[0])
      qc.cx(qReward[1], qReturn[0])
      
      qc.barrier()

      ## For Return qubit[1]
      qc.cx(t0_qReward[0], qReturn[1])
      qc.cx(t1_qReward[0], qReturn[1])
      qc.cx(qReward[0], qReturn[1])
      # For carry-in part
      qc.ccx(t0_qReward[1], t1_qReward[1], qReturn[1])
      qc.ccx(t0_qReward[1], qReward[1], qReturn[1])
      qc.ccx(t1_qReward[1], qReward[1], qReturn[1])
      qc.barrier()
      
      # For Return qubit [2]
      # when no carry-in bit from LSB
      qc.ccx(t0_qReward[0],  t1_qReward[0], qReturn[2])
      qc.ccx(t0_qReward[0], qReward[0],qReturn[2])
      qc.ccx(t1_qReward[0], qReward[0] ,qReturn[2])

      # when carry-in bit from LSB
      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      ## carry-in and single one qubit
      qc.mcx([t0_qReward[1], t1_qReward[1],t0_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], t1_qReward[1], t1_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], t1_qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1],  t1_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t1_qReward[1], qReward[1],t0_qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t1_qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[2]])

      qc.barrier()

      ### for  Return qubit [3] (carry-out qubit)
      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[3]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[3]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[3]])
      
    def __UOracle(self, qc, qState, qNextState,t0_qReward,t1_qReward, t2_qReward,qReturn0, qReturn1, qReturn2, qReturn3): 
     
    # # case 18     
     qc.x([qReturn0,qReturn1,qReturn2]) 
     qc.h(qReturn3)
     qc.mcx([t0_qReward[1], t1_qReward[0],t1_qReward[1], t2_qReward[0],t2_qReward[1], qReturn0,qReturn1,qReturn2],qReturn3) # case 18
     qc.h(qReturn3)
     qc.x([qReturn0,qReturn1,qReturn2])

     
    # self.__UDiffuser(self.qc, self.qS, self.qA, self.qSp, self.t0_qAnci, self.t1_qAnci, self.qR, self.qRt[0], self.qRt[1], self.qRt[2], self.qRt[3])
    def __UDiffuser(self, qc, qState, qAction, qNextState,t0_qReward,t1_qReward, t2_qReward,qReturn0, qReturn1, qReturn2, qReturn3):

     qc.x([qState[0], qState[1]])
     qc.h([qState[0], qState[1], qAction[0]])
     qc.x([qState[0], qState[1], qAction[0]])
     qc.barrier()
     qc.h(t0_qReward[1])
     qc.mcx([qState[0], qState[1], qAction[0]],t0_qReward[1])
     qc.h(t0_qReward[1])
     qc.barrier()
     qc.h(t1_qReward[0])
     qc.h(t1_qReward[1])
     qc.mcx([qState[0], qState[1], qAction[0]],t1_qReward[1])
     qc.mcx([qState[0], qState[1], qAction[0]],t1_qReward[0])
     qc.h(t1_qReward[0])
     qc.h(t1_qReward[1])
     qc.barrier()
     qc.h(t2_qReward[0])
     qc.h(t2_qReward[1])
     qc.mcx([qState[0], qState[1], qAction[0]],t2_qReward[1])
     qc.mcx([qState[0], qState[1], qAction[0]],t2_qReward[0])
     qc.h(t2_qReward[0])
     qc.h(t2_qReward[1])
     qc.barrier()
     qc.x([qState[0], qState[1], qAction[0]])
     qc.h([qState[0], qState[1], qAction[0]])
     qc.barrier()
     
     qc.h([qReturn0, qReturn1, qReturn2, qReturn3])
     qc.mcx([t0_qReward[1],t1_qReward[0],t1_qReward[1],t2_qReward[0],t2_qReward[1]],qReturn3)
     qc.mcx([t0_qReward[1],t1_qReward[0],t1_qReward[1],t2_qReward[0],t2_qReward[1]],qReturn2)
     qc.mcx([t0_qReward[1],t1_qReward[0],t1_qReward[1],t2_qReward[0],t2_qReward[1]],qReturn1)
     qc.mcx([t0_qReward[1],t1_qReward[0],t1_qReward[1],t2_qReward[0],t2_qReward[1]],qReturn0)
     qc.h([qReturn0, qReturn1, qReturn2, qReturn3])
    

In [ ]:
# ============================================================
# 2. ASSIGN  FOUR EXISTING CIRCUITS HERE
# ============================================================

# ===================================
# Static QMDP before Grover
# ==================================
env_R = QuantumToyEnv_Static()

__URt(env_R.qc, env_R.qR0[1], env_R.qR0[0], env_R.qR1[1], env_R.qR1[0], env_R.qR2[1], env_R.qR2[0], env_R.qRt[0], env_R.qRt[1], env_R.qRt[2], env_R.qRt[3])

static_noG_circuit = env_R.qc
static_noG_circuit.measure_all()

# print("static_noG_circuit")
# static_noG_circuit.draw('mpl', fold = -1)

# ===================================
# Static QMDP After Grover
# ===================================
env_R_2= QuantumToyEnv_Static() 

### implement return function to "env" MDP circuit
__URt(env_R_2.qc, env_R_2.qR0[1], env_R_2.qR0[0], env_R_2.qR1[1],env_R_2.qR1[0],env_R_2.qR2[1],env_R_2.qR2[0],env_R_2.qRt[0],env_R_2.qRt[1],env_R_2.qRt[2],env_R_2.qRt[3]) 

### implement oracle
env_R_2.qc = env_R_2.qc.compose(oracle, env_R_2.qc.qubits)
env_R_2.qc.barrier()

# implement diffuser
env_R_2.qc = env_R_2.qc.compose(diffuser, env_R_2.qc.qubits)
static_G_circuit =  env_R_2.qc
static_G_circuit.measure_all()

# print("static_G_circuit")
# static_G_circuit.draw('mpl', fold = -1)

# ===================================
# Dynamic QMDP before Grover
# ==================================

dynamic_qMDP_return = QuantumToyEnv_Dynamic()
dynamic_noG_circuit = dynamic_qMDP_return.qc

# print("dynamic_noG_circuit")
# dynamic_noG_circuit.draw('mpl', fold = -1)

# ===================================
# Dynamic QMDP after Grover
# ==================================

dynQMDP_gro_1000= QuantumToyEnv_Dynamic_G()
dynamic_G_circuit = dynQMDP_gro_1000.qc

# print("dynamic_G_circuit")
# dynamic_G_circuit.draw('mpl', fold = -1)


In [ ]:
# print("static_noG_circuit")
# static_noG_circuit.draw('mpl', fold = -1)

# print("static_G_circuit")
# static_G_circuit.draw('mpl', fold = -1)

# print("dynamic_noG_circuit")
# dynamic_noG_circuit.draw('mpl', fold = -1)

# print("dynamic_G_circuit")
# dynamic_G_circuit.draw('mpl', fold = -1)


In [ ]:
# ============================================================
# 3. IDEAL AER SIMULATOR
# ============================================================

simulator = AerSimulator()


# ============================================================
# 4. TRANSPILE EACH CIRCUIT ONCE
# ============================================================
#
# Important:
# Keep the transpiled circuit fixed across the 10 runs.
# Only seed_simulator is changed.
#
# A fixed transpiler seed prevents compilation randomness from
# being mixed with shot-sampling randomness.
# ============================================================

TRANSPILE_SEED = 12345

static_noG_t = transpile(
    static_noG_circuit,
    simulator,
    optimization_level=0,
    seed_transpiler=TRANSPILE_SEED
)

static_G_t = transpile(
    static_G_circuit,
    simulator,
    optimization_level=0,
    seed_transpiler=TRANSPILE_SEED
)

dynamic_noG_t = transpile(
    dynamic_noG_circuit,
    simulator,
    optimization_level=0,
    seed_transpiler=TRANSPILE_SEED
)

dynamic_G_t = transpile(
    dynamic_G_circuit,
    simulator,
    optimization_level=0,
    seed_transpiler=TRANSPILE_SEED
)

In [ ]:
# ============================================================
# Helper function (Function to reverse the four most significant bits)
# ============================================================

def reverse_qubit_order_msb(value):
    if len(value) >= 4:
        most_significant_four = value[:4]
        rest = value[4:]
        reversed_most_significant_four = most_significant_four[::-1]
        return reversed_most_significant_four + rest
    return value


# ============================================================
# Static QMDP bit-order correction
# ============================================================

def correct_static_qubit_order(value):
    if len(value) >= 21:
        # Return
        rt = value[:4]
        # t = 2
        t2 = value[4:11]
        t2_msb = reverse_qubit_order_msb(t2)
        # t = 1
        t1 = value[11:18]
        t1_msb = reverse_qubit_order_msb(t1)
        # t = 0
        t0 = value[18:]
        t0_msb = reverse_qubit_order_msb(t0)
        return rt + t2_msb + t1_msb + t0_msb

    return value

In [ ]:
# ============================================================
# 5. HELPER FUNCTIONS
# ============================================================

def clean_counts(counts):
    """
    Remove spaces that Qiskit may insert between classical
    registers.

    Example:
        '1000 1111111 1111110 1010000'
    becomes
        '1000111111111111101010000'

    IMPORTANT:
    This does NOT reverse bit order.

    It assumes the circuit measurement ordering is already the
    same ordering used in your corrected trajectory files.
    """
    cleaned = {}

    for bitstring, count in counts.items():
        key = bitstring.replace(" ", "")
        cleaned[key] = cleaned.get(key, 0) + count

    return cleaned


def get_target_metrics(counts, shots=SHOTS):
    """
    Return:
        T-143 count
        T-151 count
        combined target count
        target probability
    """

    counts = clean_counts(counts)

    n143 = counts.get(T143, 0)
    n151 = counts.get(T151, 0)

    n_target = n143 + n151

    # Reviewer-requested probability uses ALL original shots
    p_target = n_target / shots

    return n143, n151, n_target, p_target


def run_one_seed(circuit, seed):
    """
    Execute one ideal Aer simulation for the specified
    simulator seed.
    """

    result = simulator.run(
        circuit,
        shots=SHOTS,
        seed_simulator=seed
    ).result()

    counts = result.get_counts()
   
    return counts
  

In [ ]:
# ============================================================
# 6. RUN THE 10 INDEPENDENT SEEDS
# ============================================================

all_results = []
dynamic_all_counts_G = []

for seed in SEEDS:

    print(f"Running seed {seed} ...")

    # --------------------------------------------------------
    # Static QMDP without Grover
    # --------------------------------------------------------

    counts_static_noG = run_one_seed(static_noG_t, seed)

    ( static_noG_T143,  static_noG_T151, static_noG_target, static_noG_prob) = get_target_metrics(counts_static_noG)


    # --------------------------------------------------------
    # Static QMDP with Grover
    # --------------------------------------------------------

    counts_static_G = run_one_seed( static_G_t, seed)

    ( static_G_T143, static_G_T151, static_G_target, static_G_prob) = get_target_metrics(counts_static_G)


    # --------------------------------------------------------
    # Dynamic QMDP without Grover
    # --------------------------------------------------------

    counts_dynamic_noG = run_one_seed( dynamic_noG_t, seed )

    ( dynamic_noG_T143, dynamic_noG_T151,  dynamic_noG_target,  dynamic_noG_prob ) = get_target_metrics(counts_dynamic_noG)


    # --------------------------------------------------------
    # Dynamic QMDP with Grover
    # --------------------------------------------------------

    counts_dynamic_G = run_one_seed( dynamic_G_t, seed )

    ( dynamic_G_T143, dynamic_G_T151,  dynamic_G_target, dynamic_G_prob ) = get_target_metrics(counts_dynamic_G)
    
    counts_dynamic_G_qubit_order= {correct_static_qubit_order(key): value for key, value in counts_dynamic_G.items()}  ## Next state qubits and reward qubits are in correct order.
    
    dynamic_all_counts_G.append(counts_dynamic_G_qubit_order)

    # --------------------------------------------------------
    # Amplification gain
    # --------------------------------------------------------

    if static_noG_prob > 0:
        static_gain = static_G_prob / static_noG_prob
    else:
        static_gain = np.nan


    if dynamic_noG_prob > 0:
        dynamic_gain = dynamic_G_prob / dynamic_noG_prob
    else:
        dynamic_gain = np.nan


    # Relative probability increase (%)
    if static_noG_prob > 0:
        static_relative_increase = (
            (static_G_prob - static_noG_prob)
            / static_noG_prob
        ) * 100
    else:
        static_relative_increase = np.nan


    if dynamic_noG_prob > 0:
        dynamic_relative_increase = (
            (dynamic_G_prob - dynamic_noG_prob)
            / dynamic_noG_prob
        ) * 100
    else:
        dynamic_relative_increase = np.nan


    # --------------------------------------------------------
    # Save this seed
    # --------------------------------------------------------

    all_results.append({

        "seed": seed,

        # Static no Grover
        "static_noG_T143": static_noG_T143,
        "static_noG_T151": static_noG_T151,
        "static_noG_target_count": static_noG_target,
        "static_noG_probability": static_noG_prob,

        # Static Grover
        "static_G_T143": static_G_T143,
        "static_G_T151": static_G_T151,
        "static_G_target_count": static_G_target,
        "static_G_probability": static_G_prob,

        # Static amplification
        "static_amplification_gain": static_gain,
        "static_relative_increase_percent":
            static_relative_increase,

        # Dynamic no Grover
        "dynamic_noG_T143": dynamic_noG_T143,
        "dynamic_noG_T151": dynamic_noG_T151,
        "dynamic_noG_target_count": dynamic_noG_target,
        "dynamic_noG_probability": dynamic_noG_prob,

        # Dynamic Grover
        "dynamic_G_T143": dynamic_G_T143,
        "dynamic_G_T151": dynamic_G_T151,
        "dynamic_G_target_count": dynamic_G_target,
        "dynamic_G_probability": dynamic_G_prob,

        # Dynamic amplification
        "dynamic_amplification_gain": dynamic_gain,
        "dynamic_relative_increase_percent":
            dynamic_relative_increase
    })

In [ ]:
total_counts = sum(
    sum(d.values()) for d in dynamic_all_counts_G
)
print("Total counts:", total_counts)

total_length = sum(
    len(d.keys()) for d in dynamic_all_counts_G
)
print("Total trajectories:", total_length)

In [ ]:
print(type(dynamic_all_counts_G))
print(type(dynamic_all_counts_G[0]))
print(len(dynamic_all_counts_G))

In [ ]:
len(dynamic_all_counts_G[0])

In [ ]:
dynamic_all_counts_G

In [ ]:
# ============================================================
# 7. CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(all_results)

print("\n")
print("=" * 90)
print("10-SEED RESULTS")
print("=" * 90)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print(df)


In [ ]:
# ============================================================
# 8. MORE READABLE SEED-BY-SEED TABLE
# ============================================================

display_columns = [

    "seed",

    "static_noG_target_count",
    "static_noG_probability",

    "static_G_target_count",
    "static_G_probability",

    "static_amplification_gain",

    "dynamic_noG_target_count",
    "dynamic_noG_probability",

    "dynamic_G_target_count",
    "dynamic_G_probability",

    "dynamic_amplification_gain"
]

print("\n")
print("=" * 90)
print("COMPACT RESULTS")
print("=" * 90)

print(df[display_columns].to_string(index=False))


In [ ]:
# ============================================================
# 9. STATISTICAL SUMMARY FUNCTION
# ============================================================

def statistical_summary(values, confidence=0.95):

    values = np.asarray(values, dtype=float)

    # Remove NaN values if any denominator was zero
    values = values[~np.isnan(values)]

    n = len(values)

    mean = np.mean(values)

    # Sample standard deviation
    std = np.std(values, ddof=1)

    # Standard error
    sem = std / np.sqrt(n)

    # Student-t critical value
    alpha = 1 - confidence
    t_critical = t.ppf(
        1 - alpha / 2,
        df=n - 1
    )

    margin = t_critical * sem

    ci_low = mean - margin
    ci_high = mean + margin

    return {
        "n": n,
        "mean": mean,
        "std": std,
        "ci_low": ci_low,
        "ci_high": ci_high
    }


In [ ]:
# ============================================================
# 10. CALCULATE MEAN ± SD AND 95% CI
# ============================================================

metrics = {

    "Static no-Grover target probability":
        df["static_noG_probability"],

    "Static Grover target probability":
        df["static_G_probability"],

    "Static amplification gain":
        df["static_amplification_gain"],

    "Dynamic no-Grover target probability":
        df["dynamic_noG_probability"],

    "Dynamic Grover target probability":
        df["dynamic_G_probability"],

    "Dynamic amplification gain":
        df["dynamic_amplification_gain"]
}


summary_rows = []

print("\n")
print("=" * 90)
print("MEAN ± STANDARD DEVIATION AND 95% CONFIDENCE INTERVAL")
print("=" * 90)


for metric_name, values in metrics.items():

    stats = statistical_summary(values)

    summary_rows.append({
        "Metric": metric_name,
        "N": stats["n"],
        "Mean": stats["mean"],
        "SD": stats["std"],
        "95% CI lower": stats["ci_low"],
        "95% CI upper": stats["ci_high"]
    })

    print(f"\n{metric_name}")

    print(
        f"Mean ± SD = "
        f"{stats['mean']:.6f} ± "
        f"{stats['std']:.6f}"
    )

    print(
        f"95% CI = "
        f"[{stats['ci_low']:.6f}, "
        f"{stats['ci_high']:.6f}]"
    )


summary_df = pd.DataFrame(summary_rows)

In [ ]:
# ============================================================
# 11. PRINT PROBABILITIES AS PERCENTAGES
# ============================================================

print("\n")
print("=" * 90)
print("PROBABILITY RESULTS AS PERCENTAGES")
print("=" * 90)


probability_columns = {

    "Static no Grover":
        "static_noG_probability",

    "Static Grover":
        "static_G_probability",

    "Dynamic no Grover":
        "dynamic_noG_probability",

    "Dynamic Grover":
        "dynamic_G_probability"
}


for name, column in probability_columns.items():

    stats = statistical_summary(df[column])

    print(
        f"\n{name}: "
        f"{stats['mean'] * 100:.3f}% "
        f"± {stats['std'] * 100:.3f}%"
    )

    print(
        f"95% CI: "
        f"[{stats['ci_low'] * 100:.3f}%, "
        f"{stats['ci_high'] * 100:.3f}%]"
    )

In [ ]:
# ============================================================
# 12. AMPLIFICATION GAIN SUMMARY
# ============================================================

static_gain_stats = statistical_summary(
    df["static_amplification_gain"]
)

dynamic_gain_stats = statistical_summary(
    df["dynamic_amplification_gain"]
)


print("\n")
print("=" * 90)
print("AMPLIFICATION GAIN")
print("=" * 90)

print(
    "\nStatic QMDP:\n"
    f"{static_gain_stats['mean']:.3f}x "
    f"± {static_gain_stats['std']:.3f}x\n"
    f"95% CI = "
    f"[{static_gain_stats['ci_low']:.3f}x, "
    f"{static_gain_stats['ci_high']:.3f}x]"
)

print(
    "\nDynamic QMDP:\n"
    f"{dynamic_gain_stats['mean']:.3f}x "
    f"± {dynamic_gain_stats['std']:.3f}x\n"
    f"95% CI = "
    f"[{dynamic_gain_stats['ci_low']:.3f}x, "
    f"{dynamic_gain_stats['ci_high']:.3f}x]"
)



In [ ]:
# ============================================================
# 13. OPTIONAL: RELATIVE INCREASE
# ============================================================

static_relative_stats = statistical_summary(
    df["static_relative_increase_percent"]
)

dynamic_relative_stats = statistical_summary(
    df["dynamic_relative_increase_percent"]
)


print("\n")
print("=" * 90)
print("RELATIVE INCREASE IN TARGET PROBABILITY")
print("=" * 90)

print(
    "\nStatic QMDP:\n"
    f"{static_relative_stats['mean']:.2f}% "
    f"± {static_relative_stats['std']:.2f}%"
)

print(
    "\nDynamic QMDP:\n"
    f"{dynamic_relative_stats['mean']:.2f}% "
    f"± {dynamic_relative_stats['std']:.2f}%"
)

In [ ]:
# ============================================================
# 14. SANITY CHECK
# ============================================================

print("\n")
print("=" * 90)
print("SANITY CHECK")
print("=" * 90)

print(
    "Total target counts across 10 seeds:"
)

print(
    "Static no Grover:",
    df["static_noG_target_count"].sum()
)

print(
    "Static Grover:",
    df["static_G_target_count"].sum()
)

print(
    "Dynamic no Grover:",
    df["dynamic_noG_target_count"].sum()
)

print(
    "Dynamic Grover:",
    df["dynamic_G_target_count"].sum()
)


# Warn if target strings were never found

for column in [
    "static_noG_target_count",
    "static_G_target_count",
    "dynamic_noG_target_count",
    "dynamic_G_target_count"
]:

    if df[column].sum() == 0:

        print(
            f"\nWARNING: {column} is zero for all seeds."
        )

        print(
            "Check the classical-register ordering / "
            "bit-string correction used in your original code."
        )


In [ ]:
# ============================================================
# 15. SAVE RESULTS
# ============================================================

df.to_csv(
    "reviewer1_point9_10seed_target_metrics.csv",
    index=False
)

summary_df.to_csv(
    "reviewer1_point9_10seed_summary.csv",
    index=False
)

print("\n")
print("=" * 90)
print("FILES SAVED")
print("=" * 90)

print(
    "reviewer1_point9_10seed_target_metrics.csv"
)

print(
    "reviewer1_point9_10seed_summary.csv"
)

In [ ]:
for seed in [1, 10, 100, 1000, 9999]:

    result = simulator.run(
        dynamic_noG_t,
        shots=1024,
        seed_simulator=seed
    ).result()

    counts = result.get_counts()

    corrected = clean_counts(counts)

    print("\nSeed:", seed)
    print("T-143:", corrected.get(T143, 0))
    print("T-151:", corrected.get(T151, 0))

    print("Number of unique outcomes:", len(corrected))

In [ ]:
result1 = simulator.run(
    dynamic_noG_t,
    shots=1024,
    seed_simulator=123
).result().get_counts()

result2 = simulator.run(
    dynamic_noG_t,
    shots=1024,
    seed_simulator=98765
).result().get_counts()

print("Raw count dictionaries identical?",
      result1 == result2)